In [1]:
import os
import sys
path_cwd = os.getcwd()
sys.path.append(os.path.dirname(path_cwd))
sys.path.append(os.path.dirname(os.path.dirname(path_cwd)))

# Q1

In [15]:
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader, chunk_documents
import json
from minsearch import Index
from openai import OpenAI
import rag_helper as rh

In [3]:
load_dotenv()

True

In [3]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [5]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [6]:
len(documents)

72

In [7]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [8]:
documents[0].keys()

dict_keys(['content', 'filename'])

## save documents

In [ ]:
data_path = os.path.join(os.path.dirname(path_cwd),"data")

In [24]:
if not os.path.exists(data_path):
    os.makedirs(data_path)

In [ ]:
documents_path = os.path.join(data_path,"homework_documents.json")

In [27]:
json_str = json.dumps(documents, indent=4)
with open(documents_path, "w",encoding="utf-8") as f:
    f.write(json_str)

## load documents

In [4]:
data_path = os.path.join(os.path.dirname(path_cwd),"data")
documents_path = os.path.join(data_path,"homework_documents.json")

In [5]:
with open(documents_path, "r", encoding="utf-8") as f:
    documents = json.load(f)

# call to gpt models

In [6]:
openai_client = OpenAI()

In [14]:
def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text, response

In [15]:
text, oresponse = llm("Hey, what's up?")

In [16]:
type(oresponse)

openai.types.responses.response.Response

In [17]:
dir(oresponse)

['__abstractmethods__',
 '__annotate_func__',
 '__annotations__',
 '__annotations_cache__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_extra_info__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_parent_namespace__',
 '__pydan

In [18]:
dir(oresponse.usage)

['__abstractmethods__',
 '__annotate_func__',
 '__annotations__',
 '__annotations_cache__',
 '__class__',
 '__class_getitem__',
 '__class_vars__',
 '__copy__',
 '__deepcopy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__fields__',
 '__fields_set__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__get_pydantic_core_schema__',
 '__get_pydantic_json_schema__',
 '__getattr__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__pretty__',
 '__private_attributes__',
 '__pydantic_complete__',
 '__pydantic_computed_fields__',
 '__pydantic_core_schema__',
 '__pydantic_custom_init__',
 '__pydantic_decorators__',
 '__pydantic_extra__',
 '__pydantic_extra_info__',
 '__pydantic_fields__',
 '__pydantic_fields_set__',
 '__pydantic_generic_metadata__',
 '__pydantic_init_subclass__',
 '__pydantic_on_complete__',
 '__pydantic_parent_namespace__',
 '__pydan

# Q2

In [7]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [8]:
question = "How does the agentic loop keep calling the model until it stops?"

In [8]:
search_results = index.search(
    question,
    num_results=1
)

search_results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

# Q3

In [63]:
len(search_results[0]["content"])

10121

In [9]:
query='How does the agentic loop keep calling the model until it stops?'

In [10]:
rag_base = rh.RAGBase(index=index,llm_client=openai_client)

In [11]:
output = rag_base.rag(query)

In [14]:
output.answer

'It keeps calling the model inside a `while True` loop, and after each response it checks whether any `function_call` items were returned.\n\n- If there are function calls, the code runs the tool, appends the tool output to `messages`, and loops again.\n- If there are no function calls, it breaks and returns the final answer.\n\nThe key exit condition is:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nSo the model is called repeatedly until it stops asking for tools.'

In [13]:
output.output.usage.input_tokens

7136

# Q4

In [16]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [24]:
len(chunks[0]['content']), len(chunks[-1]['content'])

(2000, 1348)

In [21]:
len(chunks)

295